<a href="https://colab.research.google.com/github/ced-sys/AI-N-ML/blob/main/Old_Assyrian_Gemma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!huggingface-cli login

In [ ]:
import pandas as pd
import numpy as np
import re
import torch
from pathlib import Path
from tqdm.auto import tqdm
from typing import List, Dict, Tuple
import warnings
import zipfile
import pickle
warnings.filterwarnings('ignore')

In [ ]:
!pip install evaluate
!pip install -q bitsandbytes>=0.46.1

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import gc
gc.collect()

In [ ]:
from transformers import(
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig # Added this import
)

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
import evaluate
from sklearn.model_selection import train_test_split

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
  print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
def load_data_from_drive(
    drive_zip_path:str="/content/drive/MyDrive/Data Folder/data.zip",
    extract_to: Path=Path('/content/data')
)-> Path:

  from google.colab import drive as colab_drive

  mount_point=Path("/content/drive")
  if not(mount_point/ "MyDrive").exists():
    print("\nMounting Google Drive...")
    colab_drive.mount(str(mount_point))
  else:
    print("\nGoogle Drive already mounted.")

  zip_path=mount_point/drive_zip_path

  if not zip_path.exists():
    raise FileNotFoundError(
        f"\nCould not find zip at: {zip_path}"
    )

  print(f"\nFound: {zip_path}")
  size_mb=zip_path.stat().st_size/ (1024* 1024)
  print(f"Size: {size_mb:.1f} MB")

  extract_to.mkdir(exist_ok=True, parents=True)
  print(f'Extracting to {extract_to}...')

  with zipfile.ZipFile(zip_path, "r") as zf:
    members=zf.namelist()
    print(f" {len(members)} files in archive")
    for member in tqdm(members, desc="Extracting"):
      zf.extract(member, extract_to)


  top_level=list(extract_to.iterdir())
  if len(top_level)==1 and top_level[0].is_dir():
    subdir=top_level[0]
    print(f"\nFlattening subdirectory: {subdir.name}/")
    for f in subdir.iterdir():
      f.rename(extract_to / f.name)
    subdir.rmdir()

  required=[
      "train.csv",
      "test.csv",
      "published_texts.csv",
      "publications.csv",
      "OA_Lexicon_eBL.csv",
      "Sentences_Oare_FirstWord_LinNum.csv",
  ]
  missing=[]
  for fname in required:
    fpath=extract_to/fname
    if fpath.exists():
      mb=fpath.stat().st_size/ (1024*1024)
      print(f" OK {fname:<40s} ({mb:>7.1f}) MB")
    else:
      missing.append(fname)
      print(f" -- {fname:<40s} (MISSING)")

  if missing:
    print(f"WARNING: {len(missing)} file(s) missing: {', '.join(missing)}")
  else:
    print(f"ALL {len(required)} required files found.")

  return extract_to

In [ ]:
DRIVE_ZIP_PATH='/content/drive/MyDrive/Data Folder/data.zip'

In [ ]:
DATA_DIR=Path('/content/data')
OUTPUT_DIR=Path('/content/drive/MyDrive/old_assyrian_models')
KAGGLE_EXPORT_DIR=Path('/content/drive/MyDrive/old_assyrian_kaggle')

MODEL_NAME="google/gemma-2-2b-it"
MAX_LENGTH=128

NUM_EPOCHS=2
BATCH_SIZE=1
GRADIENT_ACCUMULATION_STEPS=32
LEARNING_RATE=2e-4
LORA_RANK=4
LORA_ALPHA=8

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
KAGGLE_EXPORT_DIR.mkdir(exist_ok=True, parents=True)

In [ ]:
class AkkadianNormalizer:

  def __init__(self):
    self.vowel_map={
        '\u00e1':'a2',
        '\u00e0':'a3',
        '\u00e2':'a2',
        '\u00e9':'e2',
        '\u00e8':'e3',
        '\u00ea':'e2',
        '\u00ed':'i2',
        '\u00ec':'i3',
        '\u00ee':'i2',
        '\u00fa':'u2',
        '\u00f9':'u3',
        '\u00fb':'u2',
    }

    self.consonant_map={
        '\u0161':'sz',
        '\u0160':'SZ',
        '\u1e63':'S',
        '\u1e62':'S',
        '\u1e6d':'t',
        '\u1e6c':'T',
        '\u1e2b':'h',
        '\u1e2a':'H',
    }

    self.bracket_patterns=[
        (r'\u02f9([^\u02fa]+)\u02fa', r'\1'),
        (r'<([^>]+)>', r'\1'),
        (r'<<[^>]*>>', ''),
        (r'\[([^\]]+)\]', r'\1')
    ]

    self.lacuna_patterns=[
        (r'\[\.\.\.\s*\.\.\.\]', '[...]'),
        (r'\[\s*x\s*\]', '[x]'),
        (r'\.\.\.', '[...]'),
        (r'\[([^\]]{2,})\]', '[...]'),
        (r'\[([a-z0-9])\]', '[x]')
    ]

  def normalize_diacritics(self, text: str)-> str:
    for old, new in {**self.vowel_map, **self.consonant_map}.items():
      text=text.replace(old, new)
    return text

  def clean_brackets(self, text: str)->str:
    for pattern, replacement in self.bracket_patterns:
      text=re.sub(pattern, replacement, text)
    return text

  def standardize_lacunae(self, text: str)->str:
    for pattern, replacement in self.lacuna_patterns:
      text=re.sub(pattern, replacement, text)
    return text

  def remove_line_numbers(self, text: str)->str:
    text=re.sub(r'^\d+\'*\s*', '', text, flags=re.MULTILINE)
    text=re.sub(r'\s+\d+\'*\s*', '', text)
    return text

  def clean_punctuation(self, text: str)-> str:
    text=re.sub(r'[!?/]', '', text)
    text=re.sub(r'(?<!\d):(?!\d)', '', text)
    return text

  def normalize(self, text: str)->str:
    if pd.isna(text):
      return ""

    text=str(text)
    text=self.normalize_diacritics(text)
    text=self.clean_brackets(text)
    text=self.standardize_lacunae(text)
    text=self.remove_line_numbers(text)
    text=self.clean_punctuation(text)

    text=re.sub(r'\s+', '', text)
    text=text.strip()

    return text

In [ ]:
normalizer=AkkadianNormalizer()

test_text="a-na {d}A\u0161\u0161ur \u00e1-bi-ia q\u00ed-b\u00ed-ma [... ...] \u0161u-ma"
print(f"Original: {test_text}")
print(f"Normalized: {normalizer.normalize(test_text)}")

In [ ]:
def load_and_preprocess_training_data(data_dir: Path)-> pd.DataFrame:
  train_df=pd.read_csv(data_dir / 'train.csv')
  print(f"Training samples: {len(train_df)}")

  train_df['transliteration_clean']=train_df['transliteration'].apply(normalizer.normalize)
  train_df['translation_clean']=train_df['translation'].fillna('')

  train_df=train_df[train_df['translation_clean'].str.len()>0]
  print(f"After cleaning: {len(train_df)} samples")

  sample_idx=0
  print("CLEANED SAMPLE")
  print(f"Old Assyrian: {train_df.iloc[sample_idx]['transliteration_clean'][:200]}...")
  print(f"English: {train_df.iloc[sample_idx]['translation_clean'][:200]}...")

  return train_df

In [ ]:
def extract_translation_pairs(page_text: str, max_pairs: int=10)-> List[Tuple[str, str]]:
  pairs=[]
  lines=page_text.split('\n')

  i=0
  while i<len(lines)-1 and len(pairs)<max_pairs:
    line=lines[i].strip()

    if ('-' in line and len(line.split())> 2 and re.search(r'[a-z]{2,}-[a-z]{2,}', line)):
      translation=""
      for j in range(i+1, min(i+4, len(lines))):
        next_line=lines[j].strip()

        if (next_line and next_line[0].isupper() and '-' not in next_line):
          translation=next_line
          break

      if translation:
        akkadian=normalizer.normalize(line)
        pairs.append((akkadian, translation))
        i=j+1
        continue

    i+=1

  return pairs


In [ ]:
def extract_synthetic_data(data_dir: Path, num_pages: int=100)-> List[Tuple[str, str]]:
  try:
    publications_df=pd.read_csv(data_dir/ 'publications.csv')
    print(f"Publications: {len(publications_df)} pages")
    print(f"Pages with Akkadian: {publications_df['has_akkadian'].sum()}")

    synthetic_pairs=[]

    akkadian_pages=publications_df[publications_df['has_akkadian']==True].sample(
        min(num_pages, publications_df['has_akkadian'].sum()),
        random_state=42
    )

    print(f"Extracting pairs from {len(akkadian_pages)} pages...")
    for _, row in tqdm(akkadian_pages.iterrows(), total=len(akkadian_pages), desc="Extracting Pairs"):
      pairs=extract_translation_pairs(row['page_text'])
      synthetic_pairs.extend(pairs)

    print(f"Extracted {len(synthetic_pairs)} synthetic pairs")

    print("\nSynthetic pair samples:")
    for i, (akk, eng) in enumerate(synthetic_pairs[:3], 1):
      print(f"\n{i}.")
      print(f"OA: {akk[:100]}...")
      print(f" EN: {eng[:100]}...")

    return synthetic_pairs

  except FileNotFoundError:
    print("Publication files not found - using only train.csv")
    return []

In [ ]:
def create_training_dataset(train_df: pd.DataFrame, synthetic_pairs: List[Tuple[str, str]]) -> Tuple[List[Dict], List[Dict]]:
  training_data=[]

  for _, row in train_df.iterrows():
    training_data.append({
        'akkadian':row['transliteration_clean'],
        'english':row['translation_clean'],
        'source':'train.csv'
    })

  for akk, eng in synthetic_pairs:
    if len(akk.split())> 2 and len(eng.split())>2:
      training_data.append({
          'akkadian':akk,
          'english':eng,
          'source':'synthetic'
        })

  print(f"\nTotal training samples: {len(training_data)}")
  print(f" - Original: {sum(1 for d in training_data if d['source']=='train.csv')}")
  print(f" -Synthetic: {sum(1 for d in training_data if d['source']=='synthetic')}")

  train_data, val_data=train_test_split(
      training_data,
      test_size=0.1,
      random_state=42
  )

  print(f"\nTraining: {len(train_data)}")
  print(f"Validation: {len(val_data)}")

  return train_data, val_data

In [ ]:
def setup_model_and_tokenizer(model_name: str):
  from transformers import BitsAndBytesConfig

  print(f"\nLoading {model_name}.. in 4-bit (NF4)...")

  bnb_config=BitsAndBytesConfig(
      load_in_4bit=True,
      bnb_4bit_quant_type='nf4',
      bnb_4bit_compute_dtype=torch.float16,
      bnb_4bit_use_double_quant=True,
  )

  tokenizer=AutoTokenizer.from_pretrained(model_name)
  tokenizer.pad_token=tokenizer.eos_token

  model=AutoModelForCausalLM.from_pretrained(
      model_name,
      quantization_config=bnb_config,
      device_map="auto",
  )

  model.gradient_checkpointing_enable()

  model=prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

  lora_config=LoraConfig(
      r=4,
      lora_alpha=8,
      target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
      lora_dropout=0.05,
      bias="none",
      task_type="CAUSAL_LM"
  )

  model=get_peft_model(model, lora_config)
  model.print_trainable_parameters()

  return model, tokenizer


In [ ]:
def format_prompt(akkadian:str, english: str=None, tokenizer=None)-> str:
  messages=[
      {
          "role":"user",
          "content":(
              "Translate this Old Assyrian cuneiform transliteration to English:\n\n"
              f"{akkadian}"
          )
      }
  ]

  if tokenizer is not None:
    prompt=tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )
    if english is not None:
      prompt+=f"{english}<end_of_turn>"
  else:
    prompt=(
        "<start_of_turn>user\n"
        "Translate this Old Assyrian cuneiform transliteratio to English:\n\n"
        f"{akkadian}<end_of_turn>\n"
        "<start_of_turn>\n"
    )
    if english is not None:
      prompt+=f"{english}<end_of_turn>"

  return prompt

In [ ]:
def preprocess_function(examples, tokenizer):
  model_inputs={"input_ids":[], "attention_mask":[], "labels":[]}

  for akk, eng in zip(examples['akkadian'], examples['english']):
    full_text=(
        f"<start_of_turn>user\n"
        f"Translate this Old Assyrian cuneiform transliteration to English:\n\n"
        f"{akk}<end_of_turn>\n"
        f"{eng}<end_of_turn>"
    )

    full_tokens=tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length',
        return_tensors=None
    )

    prompt_text=(
        f"<start_of_turn>user\n"
        f"Translate this Old Assyrian cuneiform transliteration to English:\\n"
        f"{akk}<end_of_turn>\n"
        f"<start_of_turn>\n"
    )

    prompt_tokens=tokenizer(prompt_text, add_special_tokens=False)
    prompt_length=len(prompt_tokens["input_ids"])

    labels=full_tokens["input_ids"].copy()

    for i in range(min(prompt_length, len(labels))):
      labels[i]=-100

    for i in range(len(labels)):
      if labels[i]==tokenizer.pad_token_id:
        labels[i]=-100

    model_inputs["input_ids"].append(full_tokens["input_ids"])
    model_inputs["attention_mask"].append(full_tokens["attntion_mask"])
    model_inputs["labels"].append(labels)

  return model_inputs

In [ ]:
def prepare_datasets(train_data: List[Dict], val_data: List[Dict], tokenizer):
  train_dataset=Dataset.from_list(train_data)
  val_dataset=Dataset.from_list(val_data)

  train_dataset=train_dataset.map(
      lambda x: preprocess_function(x, tokenizer),
      batched=True,
      remove_columns=val_dataset.column_names,
      desc='Tokenizing validation data'
  )

  val_dataset=val_dataset.map(
      lambda x: preprocess_function(x, tokenizer),
      batched=True,
      remove_columns=val_dataset.column_names,
      desc="Tokenisin validation data"
  )

  print(f"Train dataset: {len(train_dataset)} examples")
  print(f"Val dataset: {len(val_dataset)} examples")

  return train_dataset, val_dataset

In [ ]:
def create_trainer(model, tokenizer, train_dataset, val_dataset, output_dir: Path):
  training_args=TrainingArguments(
      output_dir=str(output_dir),

      num_train_epochs=NUM_EPOCHS,
      per_device_train_batch_size=BATCH_SIZE,
      per_device_eval_batch_size=BATCH_SIZE,
      gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

      learning_rate=LEARNING_RATE,
      warmup_steps=100,
      weight_decay=0.01,

      logging_steps=50,
      eval_strategy="steps",
      eval_steps=200,
      save_steps=200,
      save_total_limit=3,

      fp16=True,
      optim="paged_adamw_8bit",

      load_best_model_at_end=True,
      metric_for_best_model="eval_loss",
      report_to="none",
      push_to_hub=False,
  )

  data_collator=DataCollatorForLanguageModeling(
      tokenizer=tokenizer,
      mlm=False
  )

  trainer=Trainer(
     model=model,
     args=training_args,
     train_dataset=train_dataset,
     eval_dataset=val_dataset,
     data_collator=data_collator,
  )

  return trainer

In [ ]:
!pip install sacrebleu

In [ ]:
bleu_metric=evaluate.load("bleu")
chrf_metric=evaluate.load("chrf")

In [ ]:
def compute_metrics(predictions: List[str], references: List[str])-> Dict[str, float]:
  valid_pairs=[(p, r) for p, r in zip(predictions, references) if p is not None and p.strip()]

  if not valid_pairs:
    print("WARNING: No valid predictions")
    return {'bleu':0.0, 'chrf':0.0, 'geometric_mean':0.0}

  preds, refs=zip(*valid_pairs)

  bleu_result=bleu_metric.compute(predictions=list(preds), references=[[r] for r in refs])
  chrf_result=chrf_metric.compute(predictions=list(preds), references=list(refs), word_order=2)

  bleu_score=bleu_result['bleu']*100
  chrf_score=chrf_result['score']

  geom_mean = np.sqrt(bleu_score * chrf_score)

  return{'bleu':bleu_score, 'chrf':chrf_score, 'geometric_mean': geom_mean}

In [ ]:
def generate_translation(akkadian_text: str, model, tokenizer, max_new_tokens: int=150)-> str:
  messages=[
      {
          "role":"user",
          "content":(
              "Translate this Old Assyrian cuneiform transliteration to English:\n\n"
              f"{akkadian_text}"
          )
      }
  ]

  inputs=tokenizer.apply_chat_template(
      messages,
      add_generation_prompt=True,
      tokenize=True,
      return_dict=True,
      return_tensors="pt",
  ).to(model.device)

  with torch.no_grad():
    outputs=model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

  prompt_length=inputs['input_ids'].shape[-1]
  new_tokens=outputs[0][prompt_length:]
  translation=tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

  if not translation:
    translation="Translation unavailable"

  return translation

In [ ]:
def evaluate_on_validation(model, tokenizer, val_data: List[Dict], num_samples: int=50):
  val_sample=val_data[:num_samples]

  predictions=[]
  references=[]

  print("Generating validation predictions...")
  for sample in tqdm(val_sample):
    pred=generate_translation(sample['akkadian'], model, tokenizer)
    predictions.append(pred)
    references.append(sample['english'])

  metrics=compute_metrics(predictions, references)

  print(f"BLEU: {metrics['bleu']:.2f}")
  print(f"chrF++:  {metrics['chrf']:.2f}")
  print(f"Geometric Mean: {metrics['geometric_mean']:.2f}")

  for i in range(min(3, len(predictions))):
    print(f"\n{i+1}.")
    print(f"Old Assyrian: {val_sample[i]['akkadian'][:100]}...")
    print(f"Reference: {val_sample[i]['english'][:100]}...")
    print(f"Prediction: {predictions[i][:100]}...")

  return metrics

In [ ]:
def create_submission(model, tokenizer, data_dir: Path, output_path: str='submission.csv'):
  test_df=pd.read_csv(data_dir /'test.csv')
  print(f"\nTest samples: {len(test_df)}")

  test_df['transliteration_clean']=test_df['transliteration'].apply(normalizer.normalize)

  print("Generating test predictions...")
  test_predictions=[]

  for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    pred=generate_translation(row['transliteration_clean'], model, tokenizer)
    test_predictions.append(pred)

  submission_df=pd.DataFrame({
      'id':test_df['id'],
      'translation':test_predictions
  })

  submission_df.to_csv(output_path, index=False)
  print(f"\nSubmission saved to {output_path}")

  empty=submission_df[submission_df['translation'].str.strip()=='']
  print(f"Empty translations: {len(empty)}")

  lengths=submission_df['translation'].str.len()
  print(f"\nTranslation length stats:")
  print(f" Min: {lengths.min()}")
  print(f" Max: {lengths.max()}")
  print(f" Mean: {lengths.mean():.1f}")
  print(f" Median: {lengths.median():.1f}")

  print("\nSample translations:")
  for i in range(min(3, len(submission_df))):
    row=submission_df.iloc[i]
    print(f"\n{i+1}. ID: {row['id']}")
    print(f" Translation: {row['translation'][:150]}...")

  return submission_df

In [ ]:
def main():
  data_dir=load_data_from_drive(extract_to=DATA_DIR)

  train_df=load_and_preprocess_training_data(data_dir)

  synthetic_pairs=extract_synthetic_data(data_dir, num_pages=100)

  train_data,val_data=create_training_dataset(train_df, synthetic_pairs)

  model, tokenizer=setup_model_and_tokenizer(MODEL_NAME)

  train_dataset, val_dataset=prepare_datasets(train_data, val_data, tokenizer)

  trainer=create_trainer(model, tokenizer, train_dataset, val_dataset, OUTPUT_DIR)

  trainer.train()

  final_model_path=OUTPUT_DIR / "final_model"
  trainer.save_model(final_model_path)
  tokenizer.save_pretrained(final_model_path)
  print(f"\nModel saved to {final_model_path}")

  evaluate_on_validation(model, tokenizer, val_data, num_samples=50)

  create_submission(model, tokenizer, data_dir)

In [ ]:
if __name__=="__main__":
  main()